# Donut Receipt OCR — Fine-tuning for ReceiptIQ

This notebook fine-tunes [Donut](https://github.com/clovaai/donut) (Document Understanding Transformer) to extract structured JSON from receipt images — the same schema used by ReceiptIQ.

**What you get at the end:**
- A fine-tuned model that extracts `store_name`, `receipt_date`, `total_amount`, and line items from any receipt image
- Inference cost ~$0.0008/receipt on Modal serverless vs ~$0.013/receipt with Claude

**Runtime:** ~2 hours on free Colab T4 GPU  
**Dataset:** [CORD-v2](https://huggingface.co/datasets/naver-clova-ix/cord-v2) (1,000 labeled receipts, public domain)

---
**Before starting:** Runtime → Change runtime type → T4 GPU

## 1. Setup

In [ ]:
!pip install -q transformers==4.46.1 datasets sentencepiece Pillow torch torchvision
print('Done')

In [ ]:
import json, re, math, random
from pathlib import Path

import torch
from torch.utils.data import Dataset
from PIL import Image

from transformers import (
    DonutProcessor,
    VisionEncoderDecoderModel,
    VisionEncoderDecoderConfig,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator,
)
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Quick Demo — Pretrained Donut on a Receipt

In [ ]:
# Load the already fine-tuned CORD model to see what Donut can do
demo_processor = DonutProcessor.from_pretrained('naver-clova-ix/donut-base-finetuned-cord-v2')
demo_model = VisionEncoderDecoderModel.from_pretrained('naver-clova-ix/donut-base-finetuned-cord-v2').to(device)
demo_model.eval()
print('Loaded pretrained CORD model')

In [ ]:
# Run on a sample from the CORD test set
cord_demo = load_dataset('naver-clova-ix/cord-v2', split='test[:3]')
sample_image = cord_demo[0]['image']
sample_image

In [ ]:
def run_donut_inference(image, processor, model, task_prompt='<s_cord-v2>'):
    pixel_values = processor(image, return_tensors='pt').pixel_values.to(device)
    decoder_input_ids = processor.tokenizer(
        task_prompt, add_special_tokens=False, return_tensors='pt'
    ).input_ids.to(device)

    with torch.no_grad():
        outputs = model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            max_length=model.decoder.config.max_position_embeddings,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True,
            bad_words_ids=[[processor.tokenizer.unk_token_id]],
            return_dict_in_generate=True,
        )

    seq = processor.batch_decode(outputs.sequences)[0]
    seq = seq.replace(processor.tokenizer.eos_token, '').replace(processor.tokenizer.pad_token, '')
    seq = re.sub(r'<.*?>', '', seq, count=1).strip()  # remove task token
    return processor.token2json(seq)

result = run_donut_inference(sample_image, demo_processor, demo_model)
print('Donut extracted:')
print(json.dumps(result, indent=2))

In [ ]:
# Compare with ground truth
gt = json.loads(cord_demo[0]['ground_truth'])['gt_parse']
print('Ground truth:')
print(json.dumps(gt, indent=2))

# Free demo model memory
del demo_model, demo_processor
torch.cuda.empty_cache()

## 3. Data Preparation — CORD → ReceiptIQ Schema

In [ ]:
# Load CORD-v2 (1,000 labeled receipt images)
raw_dataset = load_dataset('naver-clova-ix/cord-v2')
print(raw_dataset)
print(f"Train: {len(raw_dataset['train'])} | Validation: {len(raw_dataset['validation'])} | Test: {len(raw_dataset['test'])}")

In [ ]:
def parse_price(s):
    if not s:
        return 0.0
    cleaned = re.sub(r'[^\d.]', '', str(s).replace(',', '.'))
    parts = cleaned.split('.')
    if len(parts) > 2:
        cleaned = ''.join(parts[:-1]) + '.' + parts[-1]
    try:
        return round(float(cleaned), 2)
    except ValueError:
        return 0.0


def cord_to_receiptiq(ground_truth_str):
    try:
        data = json.loads(ground_truth_str)['gt_parse']
    except (json.JSONDecodeError, KeyError):
        return None

    items = []
    for menu_item in data.get('menu', []):
        if not isinstance(menu_item, dict):
            continue
        name = menu_item.get('nm', '').strip()
        if not name:
            continue
        cnt = menu_item.get('cnt', '1')
        quantity = parse_price(cnt) if cnt else 1.0
        quantity = max(quantity, 1.0)
        line_total = parse_price(menu_item.get('price', '0'))
        unit_price = round(line_total / quantity, 2) if quantity else line_total
        items.append({
            'item_name': name,
            'quantity': quantity,
            'unit_price': unit_price,
            'line_total': line_total,
            'category': 'General',
        })

    total_block = data.get('total', {})
    if not isinstance(total_block, dict):
        total_block = {}
    total = parse_price(total_block.get('total_price', '0'))

    store_name = data.get('nm', 'Unknown Store')
    if not isinstance(store_name, str):
        store_name = 'Unknown Store'
    store_name = store_name.strip() or 'Unknown Store'

    return {
        'store_name': store_name,
        'store_chain': store_name,
        'receipt_date': '2024-01-01',
        'total_amount': total,
        'currency': 'USD',
        'items': items,
    }


# Test the conversion
sample = raw_dataset['train'][0]
converted = cord_to_receiptiq(sample['ground_truth'])
print('Converted to ReceiptIQ schema:')
print(json.dumps(converted, indent=2))

## 4. Model & Tokenizer Setup

In [ ]:
# Load base Donut (no CORD fine-tuning — we'll train our own schema)
MODEL_ID = 'naver-clova-ix/donut-base'
TASK_TOKEN = '<s_receipt>'   # our custom task start token
MAX_LENGTH = 768             # max decoder output tokens
IMAGE_SIZE = [1280, 960]     # height x width (Donut default)

processor = DonutProcessor.from_pretrained(MODEL_ID)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID)

# Add our custom task token + ReceiptIQ JSON field tokens
new_tokens = [
    TASK_TOKEN,
    '</s_receipt>',
]
num_added = processor.tokenizer.add_special_tokens({'additional_special_tokens': new_tokens})
print(f'Added {num_added} special tokens')

# Resize model embeddings to match new vocab
model.decoder.resize_token_embeddings(len(processor.tokenizer))

# Configure image size
processor.image_processor.size = {'height': IMAGE_SIZE[0], 'width': IMAGE_SIZE[1]}
model.config.encoder.image_size = IMAGE_SIZE
model.config.decoder.max_length = MAX_LENGTH

# Set special token IDs
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids([TASK_TOKEN])[0]

print(f'Vocab size: {len(processor.tokenizer)}')
print(f'decoder_start_token_id: {model.config.decoder_start_token_id}')
print(f'Model params: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M')

## 5. PyTorch Dataset

In [ ]:
class ReceiptDataset(Dataset):
    def __init__(self, hf_split, processor, max_length=768, split='train'):
        self.processor = processor
        self.max_length = max_length
        self.split = split

        # Convert + filter out samples that don't parse
        self.samples = []
        for item in hf_split:
            converted = cord_to_receiptiq(item['ground_truth'])
            if converted and converted['items']:
                self.samples.append({
                    'image': item['image'],
                    'target': json.dumps(converted, ensure_ascii=False),
                })

        print(f'{split}: {len(self.samples)} usable samples (of {len(hf_split)})')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = sample['image'].convert('RGB')

        # Encode image
        pixel_values = self.processor(
            image, return_tensors='pt'
        ).pixel_values.squeeze(0)

        # Encode target — wrap with task tokens
        target_str = TASK_TOKEN + sample['target'] + self.processor.tokenizer.eos_token
        labels = self.processor.tokenizer(
            target_str,
            add_special_tokens=False,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        ).input_ids.squeeze(0)

        # Mask padding tokens in loss
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {'pixel_values': pixel_values, 'labels': labels}


train_dataset = ReceiptDataset(raw_dataset['train'], processor, MAX_LENGTH, 'train')
val_dataset   = ReceiptDataset(raw_dataset['validation'], processor, MAX_LENGTH, 'val')
test_dataset  = ReceiptDataset(raw_dataset['test'], processor, MAX_LENGTH, 'test')

# Verify one sample
sample = train_dataset[0]
print(f'pixel_values shape: {sample["pixel_values"].shape}')
print(f'labels shape: {sample["labels"].shape}')
print(f'Non-masked label tokens: {(sample["labels"] != -100).sum().item()}')

## 6. Training

In [ ]:
# Training configuration tuned for T4 (16GB)
training_args = Seq2SeqTrainingArguments(
    output_dir='./donut-receipt',
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch = 8
    learning_rate=3e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    fp16=True,                        # half precision saves VRAM
    gradient_checkpointing=True,      # trade speed for VRAM
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    save_total_limit=2,
    predict_with_generate=False,      # off during training for speed
    logging_steps=20,
    dataloader_num_workers=2,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=default_data_collator,
)

print(f'Steps per epoch: {len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}')
print(f'Total steps: {trainer.args.max_steps if trainer.args.max_steps > 0 else "auto"}')
print('Starting training...')

In [ ]:
# ~90 min on free Colab T4
trainer.train()
print('Training complete!')

In [ ]:
# Save model + processor together
model.save_pretrained('./donut-receipt-final')
processor.save_pretrained('./donut-receipt-final')
print('Saved to ./donut-receipt-final')

## 7. Inference & Evaluation

In [ ]:
def predict(image, model, processor, task_token=TASK_TOKEN, max_length=MAX_LENGTH):
    """Run receipt extraction on a PIL image. Returns a dict."""
    model.eval()
    pixel_values = processor(image.convert('RGB'), return_tensors='pt').pixel_values.to(device)
    decoder_input_ids = processor.tokenizer(
        task_token, add_special_tokens=False, return_tensors='pt'
    ).input_ids.to(device)

    with torch.no_grad():
        outputs = model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            max_length=max_length,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True,
            bad_words_ids=[[processor.tokenizer.unk_token_id]],
            return_dict_in_generate=True,
        )

    seq = processor.batch_decode(outputs.sequences)[0]
    # Strip special tokens and task wrapper
    seq = seq.replace(processor.tokenizer.eos_token, '').replace(processor.tokenizer.pad_token, '')
    seq = re.sub(r'^.*?' + re.escape(task_token), '', seq)  # remove up to task token
    seq = seq.strip()

    try:
        return json.loads(seq)
    except json.JSONDecodeError:
        # Try to extract JSON from partial output
        match = re.search(r'\{.*\}', seq, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
        return {'raw_output': seq, 'error': 'parse_failed'}


# Test on 5 samples from the test set
model.to(device)
correct_totals = 0
correct_items = 0
total_samples = 0

for i in range(min(5, len(test_dataset.samples))):
    sample = test_dataset.samples[i]
    pred = predict(sample['image'], model, processor)
    gt = json.loads(sample['target'])

    pred_total = pred.get('total_amount', 0)
    gt_total = gt.get('total_amount', 0)
    total_match = abs(pred_total - gt_total) < 0.05

    gt_item_names = {x['item_name'].lower() for x in gt.get('items', [])}
    pred_item_names = {x['item_name'].lower() for x in pred.get('items', [])} if 'items' in pred else set()
    item_overlap = len(gt_item_names & pred_item_names) / max(len(gt_item_names), 1)

    correct_totals += int(total_match)
    correct_items += item_overlap
    total_samples += 1

    print(f'--- Sample {i+1} ---')
    print(f'  store:      pred={pred.get("store_name","?")} | gt={gt.get("store_name","?")} ')
    print(f'  total:      pred=${pred_total:.2f} | gt=${gt_total:.2f} | match={total_match}')
    print(f'  item recall:{item_overlap:.0%} ({len(pred_item_names)}/{len(gt_item_names)} items)')
    print()

print(f'Total accuracy:   {correct_totals}/{total_samples} ({correct_totals/total_samples:.0%})')
print(f'Avg item recall:  {correct_items/total_samples:.0%}')

## 8. Upload to Hugging Face Hub (optional)

In [ ]:
# Optional: push to HF Hub so you can load it anywhere
# from huggingface_hub import login
# login()  # enter your HF token
#
# model.push_to_hub('your-username/donut-receipt-receiptiq')
# processor.push_to_hub('your-username/donut-receipt-receiptiq')
# print('Pushed to HF Hub')

# For now — download the model folder via Colab
!zip -r donut-receipt-final.zip donut-receipt-final/
from google.colab import files
files.download('donut-receipt-final.zip')

## 9. Use Your Own ReceiptIQ Data

Once you have 200+ uploaded receipts, run this section to mix your own data into the training set.

In [ ]:
# --- Export script (run on your server, not in Colab) ---
# This generates a JSONL file with {image_url, ground_truth} pairs
# from your TimescaleDB, then download here and add to training.

EXPORT_SCRIPT = '''
import psycopg2, json

conn = psycopg2.connect(os.environ["DATABASE_URL"])
cur = conn.cursor()
cur.execute(\'\'\'SELECT "imageUrl", "rawOcrText"
FROM "Receipt"
WHERE "imageUrl" IS NOT NULL
  AND "rawOcrText" IS NOT NULL
  AND source = \'upload\'
  AND "needsReview" = false
ORDER BY "createdAt" DESC
LIMIT 5000;\'\'\')

with open("receiptiq_train.jsonl", "w") as f:
    for image_url, raw_ocr in cur.fetchall():
        try:
            ocr_data = json.loads(raw_ocr)
            # Convert from Lambda OCR format to receiptIQ schema
            record = {
                "image_url": image_url,
                "ground_truth": json.dumps({
                    "store_name": ocr_data.get("store_name", "Unknown"),
                    "store_chain": ocr_data.get("store_chain", ""),
                    "receipt_date": ocr_data.get("receipt_date", "2024-01-01"),
                    "total_amount": ocr_data.get("total_amount", 0),
                    "currency": ocr_data.get("currency", "USD"),
                    "items": [
                        {
                            "item_name": i["item_name"],
                            "quantity": i["quantity"],
                            "unit_price": i["unit_price"],
                            "line_total": i["line_total"],
                            "category": i["category"],
                        }
                        for i in ocr_data.get("items", [])
                    ]
                })
            }
            f.write(json.dumps(record) + "\\n")
        except Exception as e:
            pass

print(f"Exported {cur.rowcount} receipts")
'''
print('Run the above script on your EC2/server to export your receipt data.')
print('Then upload receiptiq_train.jsonl here and use the dataset class below.')

In [ ]:
import urllib.request

class ReceiptIQDataset(Dataset):
    """Dataset loader for your own exported ReceiptIQ data."""

    def __init__(self, jsonl_path, processor, max_length=768, cloudfront_domain=None):
        self.processor = processor
        self.max_length = max_length
        self.samples = []

        with open(jsonl_path) as f:
            for line in f:
                try:
                    self.samples.append(json.loads(line.strip()))
                except json.JSONDecodeError:
                    pass

        print(f'Loaded {len(self.samples)} samples from {jsonl_path}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Download image from CloudFront URL
        with urllib.request.urlopen(sample['image_url']) as response:
            image = Image.open(response).convert('RGB')

        pixel_values = self.processor(
            image, return_tensors='pt'
        ).pixel_values.squeeze(0)

        target_str = TASK_TOKEN + sample['ground_truth'] + self.processor.tokenizer.eos_token
        labels = self.processor.tokenizer(
            target_str,
            add_special_tokens=False,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        ).input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {'pixel_values': pixel_values, 'labels': labels}


# To combine CORD + your data:
# from torch.utils.data import ConcatDataset
# your_data = ReceiptIQDataset('receiptiq_train.jsonl', processor)
# combined = ConcatDataset([train_dataset, your_data])
# Then pass combined as train_dataset to the trainer

print('ReceiptIQDataset class ready.')
print('Upload your receiptiq_train.jsonl and uncomment the lines above.')

## 10. Deploy to Modal (serverless GPU — ~$0.0008/receipt)

After downloading `donut-receipt-final.zip`, deploy like this:

In [ ]:
MODAL_DEPLOY_CODE = '''
# modal_ocr.py  — run: modal deploy modal_ocr.py
import modal, re, json
from pathlib import Path

app = modal.App("donut-receipt-ocr")
model_volume = modal.Volume.from_name("donut-receipt-model", create_if_missing=True)

image = (
    modal.Image.debian_slim()
    .pip_install("transformers==4.40.0", "torch", "torchvision", "Pillow", "sentencepiece")
)

TASK_TOKEN = "<s_receipt>"

@app.cls(image=image, gpu="T4", volumes={"/model": model_volume})
class DonutOCR:
    @modal.enter()
    def load(self):
        from transformers import DonutProcessor, VisionEncoderDecoderModel
        import torch
        self.device = "cuda"
        self.processor = DonutProcessor.from_pretrained("/model")
        self.model = VisionEncoderDecoderModel.from_pretrained("/model").to(self.device)
        self.model.eval()

    @modal.method()
    def extract(self, image_bytes: bytes) -> dict:
        from PIL import Image
        import io, torch
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.to(self.device)
        decoder_ids = self.processor.tokenizer(
            TASK_TOKEN, add_special_tokens=False, return_tensors="pt"
        ).input_ids.to(self.device)
        with torch.no_grad():
            out = self.model.generate(
                pixel_values, decoder_input_ids=decoder_ids, max_length=768,
                pad_token_id=self.processor.tokenizer.pad_token_id,
                eos_token_id=self.processor.tokenizer.eos_token_id,
                use_cache=True, return_dict_in_generate=True,
            )
        seq = self.processor.batch_decode(out.sequences)[0]
        seq = seq.replace(self.processor.tokenizer.eos_token, "").replace(self.processor.tokenizer.pad_token, "")
        seq = re.sub(r"^.*?" + re.escape(TASK_TOKEN), "", seq).strip()
        try:
            return json.loads(seq)
        except json.JSONDecodeError:
            return {"error": "parse_failed", "raw": seq[:200]}

@app.function(image=image)
@modal.web_endpoint(method="POST")
def ocr_endpoint(image_url: str):
    import urllib.request
    with urllib.request.urlopen(image_url) as r:
        image_bytes = r.read()
    return DonutOCR().extract.remote(image_bytes)
'''

# Save the Modal deployment file
with open('modal_ocr.py', 'w') as f:
    f.write(MODAL_DEPLOY_CODE)

print('modal_ocr.py written.')
print('To deploy:')
print('  1. pip install modal')
print('  2. modal token new')
print('  3. Upload model: modal volume put donut-receipt-model donut-receipt-final/ /model')
print('  4. modal deploy modal_ocr.py')
print()
print('Your endpoint will be at: https://your-username--donut-receipt-ocr-ocr-endpoint.modal.run')
print('Cost: ~$0.0008/receipt (T4 GPU, ~2s inference)')

---
## Summary

| What | Details |
|---|---|
| **Model** | `naver-clova-ix/donut-base` fine-tuned on CORD-v2 |
| **Training time** | ~90 min on free Colab T4 |
| **Training cost** | $0 (Colab free tier) |
| **Inference cost** | ~$0.0008/receipt on Modal T4 |
| **vs Claude Sonnet** | ~16× cheaper per call |
| **Next step** | Add your own receipts via `ReceiptIQDataset` + retrain monthly |

**To improve accuracy over time:**
1. Accumulate uploaded receipts in ReceiptIQ (target: 500+ images)
2. Export via the script in Section 9
3. Mix 50% CORD + 50% yours and retrain → model learns your store mix
4. At 2,000+ uploads, your data dominates → fully custom model